In [ ]:
import pandas as pd

df_smiles_ref = pd.read_csv("antidepressant_smiles.csv")
df_annotations = pd.read_csv("annotations_data.csv")
df_openfda = pd.read_csv("openfda_data.csv")

df_smiles_ref.columns = df_smiles_ref.columns.str.strip()
df_annotations.columns = df_annotations.columns.str.strip()
df_openfda.columns = df_openfda.columns.str.strip()

unique_genes = list(df_annotations["Gene"].dropna().unique())

def user_input():
    drug_input = input("Enter Antidepressnat Name: ")
    genetic_input = input("Enter Patient Genetic Profile (ex. CYP2D6 poor metabolizer)")

    #Normalizes the inputs
    drug_input = drug_input.strip().lower()
    genetic_input = genetic_input.strip().lower()

    #Gets the row with the drug profile 
    drug_row = df_smiles_ref[df_smiles_ref["Drug:"].str.lower() == drug_input]

    if drug_row.empty:
        print(f"{drug_input} is not in the list of registered drugs!")
        return

    #Gets the smile string, graph, gene, and risk score
    smiles_string = drug_row["SMILES"].values[0]
    graph_input = convert_smiles_to_graph(smiles_string)

    detected_gene = "N/A"
    for gene in unique_genes:
        if gene.lower() in genetic_input:
            detected_gene = gene
            break

    #Determines the risk score if any of the key words are in the user's input 
    risk_score = 0.0
    if any(w in genetic_input for w in ["poor", "slow", "high risk", "increased severity", "non-metabolizer",
    "deficient", "non metabolizer", "non-functional", "lof", 
    "severely reduced", "adverse reaction"]):
        risk_score = 2.0
    elif any(w in genetic_input for w in ["intermediate", "reduced", "moderate", "mildly reduced", 
        "partial function", "decreased activity"]):
        risk_score = 1.0

    #Builds the exact one-hot array shape for the gnn layer 
    gene_vector = [1 if detected_gene == g else 0 for g in unique_genes]
    full_genetic_vector = [risk_score] + gene_vector

    #Attaches this vector to the graph input object 
    graph_input.genetic_attr = torch.tensor([full_genetic_vector], dtype=torch.float)

    matched_key = next((k for k in drug_baseline_toxicity.keys() if k.lower() == drug_input), None)
    baseline_score = drug_baseline_toxicity[matched_key] if matched_key else 0.0
    
    #Running the actual model with the processed user inputs
    
    #Tells pytorch to freeze the model weights (no training)
    model.eval() 

    #turns off gradient tracking for fast inference 
    with torch.no_grad():
        #Assigns all atoms to a single graph instance 
        graph_input.batch = torch.tensor([0]*graph_input.num_nodes, dtype=torch.long)

        #Executes the netwrok layer 
        predicted_tensor = model(graph_input)
        predicted_score = max(0.0, predicted_tensor.item())

    #Gets the top 5 side effects for the drug using the openFDA data 
    drug_df = df_openfda[df_openfda["Drug:"].str.strip().str.lower() == drug_input]
    top_reactions = drug_df.sort_values(by="Count", ascending=False).head(5)
    effects_list = list(zip(top_reactions["Reaction:"], top_reactions["Count"]))

    #Displays the results
    print(f"Report for: {drug_input.upper()}")
    print(f"Inferred gene: {detected_gene} | Risk_score: {risk_score}")
    print(f"Graph complexity: {graph_input.num_nodes} Atoms | {graph_input.edge_index.shape[1] // 2} Bonds ")

    print()
    print(f"Toxicity rating: {min(predicted_score, 100.0):.1f} / 100")
    print(f"openFDA Baseline   : {baseline_score:.1f} / 100")
    print("Top 5 historical adverse events: ")
    if effects_list:
        for rank, (reaction, count) in enumerate(effects_list, 1):
            print(f"{rank}. {reaction} ({count:,} reported cases)")

user_input()
